# Proyecto 1 - Diagnóstico inicial de los datos crudos
Establecimientos educativos hasta el nivel DIVERSIFICADO (MINEDUC, todo el país)

Este notebook:
1. Carga los 22 archivos crudos (uno por departamento / Ciudad Capital) desde `data/raw/`.
2. Los une en un solo DataFrame (sin modificar los archivos originales).
3. Genera el diagnóstico inicial

In [55]:
import pandas as pd
import numpy as np
import glob, os, re
from IPython.display import display

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)

RAW_DIR = "../data/raw"
files = sorted(glob.glob(os.path.join(RAW_DIR, "*.csv")))
print(f"Archivos crudos encontrados: {len(files)}")
for f in files:
    print(" -", os.path.basename(f))

Archivos crudos encontrados: 23
 - ALTAVERAPAZ.csv
 - BAJAVERAPAZ.csv
 - CHIMALTENANGO.csv
 - CHIQUIMULA.csv
 - CIUDADCAPITAL.csv
 - ELPROGRESO.csv
 - ESCUINTLA.csv
 - GUATEMALA.csv
 - HUEHUETENANGO.csv
 - IZABAL.csv
 - JALAPA.csv
 - JUTIAPA.csv
 - PETEN.csv
 - QUETZALTENANGO.csv
 - QUICHE.csv
 - RETALHULEU.csv
 - SACATEPEQUEZ.csv
 - SANMARCOS.csv
 - SANTAROSA.csv
 - SOLOLA.csv
 - SUCHITEPEQUEZ.csv
 - TOTONICAPAN.csv
 - ZACAPA.csv


## 1. Carga y unión de los datos crudos

In [56]:
dfs = []
for f in files:
    df = pd.read_csv(f, dtype=str, encoding="utf-8-sig")
    df["__archivo_origen"] = os.path.basename(f)
    dfs.append(df)

datos = pd.concat(dfs, ignore_index=True)
print("Filas totales (incluye posibles filas vacías por archivo):", datos.shape[0])
print("Columnas:", datos.shape[1])

Filas totales (incluye posibles filas vacías por archivo): 11890
Columnas: 18


### 1.1 Filas completamente vacías (residuo de exportación de cada CSV)
Cada archivo exportado desde el portal de MINEDUC trae una fila final vacía.
Se identifico pero no se eliminan en esta etapa (el diagnóstico se realizó sobre los datos tal cual llegaron).

In [57]:
filas_vacias = datos.drop(columns="__archivo_origen").isna().all(axis=1).sum()
tabla_vacias = pd.DataFrame({
    "métrica": ["Filas completamente vacías", "Filas con contenido"],
    "valor": [filas_vacias, datos.shape[0] - filas_vacias]
})
display(tabla_vacias)

,métrica,valor
0,Filas completamente vacías,23
1,Filas con contenido,11867


## 2. Diagnóstico del estado inicial de los datos

### 2.a Número de registros y variables

In [58]:
n_filas, n_cols = datos.shape
tabla_registros = pd.DataFrame({
    "métrica": [
        "Número de registros (crudo, incluye filas vacías)",
        "Número de registros con contenido",
        "Número de variables (sin contar __archivo_origen)"
    ],
    "valor": [n_filas, n_filas - filas_vacias, n_cols - 1]
})
display(tabla_registros)
display(pd.DataFrame({"variable": [c for c in datos.columns if c != '__archivo_origen']}))

,métrica,valor
0,"Número de registros (crudo, incluye filas vacías)",11890
1,Número de registros con contenido,11867
2,Número de variables (sin contar __archivo_origen),17


,variable
0,CODIGO
1,DISTRITO
2,DEPARTAMENTO
3,MUNICIPIO
4,ESTABLECIMIENTO
5,DIRECCION
6,TELEFONO
7,SUPERVISOR
8,DIRECTOR
9,NIVEL


### 2.b Tipo de dato de cada variable

In [59]:
tabla_tipos = pd.DataFrame({
    "variable": datos.columns,
    "tipo_en_pandas": datos.dtypes.astype(str).values
})
display(tabla_tipos)

,variable,tipo_en_pandas
0,CODIGO,str
1,DISTRITO,str
2,DEPARTAMENTO,str
3,MUNICIPIO,str
4,ESTABLECIMIENTO,str
5,DIRECCION,str
6,TELEFONO,str
7,SUPERVISOR,str
8,DIRECTOR,str
9,NIVEL,str


**Nota:** pandas reporta todas las columnas como `object` (texto) porque se cargaron con
`dtype=str` a propósito, para no perder ceros a la izquierda en códigos y evitar que pandas
infiera tipos de forma incorrecta antes de limpiar. Sin embargo, esto NO significa que el tipo
lógico de cada variable sea texto: `CODIGO`, `DISTRITO`, `TELEFONO` deberían tratarse como
identificadores/códigos (texto controlado), mientras que campos como `NIVEL`, `SECTOR`, `AREA`,
`STATUS`, `MODALIDAD`, `JORNADA`, `PLAN`, `DEPARTAMENTO`, `MUNICIPIO`, `DEPARTAMENTAL` son
variables categóricas y deberían convertirse a tipo `category`.

### 2.c Cantidad y porcentaje de valores faltantes por variable

In [60]:
faltantes = datos.drop(columns="__archivo_origen").isna().sum()
faltantes_pct = (faltantes / n_filas * 100).round(2)
tabla_faltantes = pd.DataFrame({"faltantes": faltantes, "porcentaje": faltantes_pct})
tabla_faltantes = tabla_faltantes.sort_values("faltantes", ascending=False).reset_index().rename(columns={"index": "variable"})
display(tabla_faltantes)

,variable,faltantes,porcentaje
0,DIRECTOR,1755,14.76
1,TELEFONO,969,8.15
2,SUPERVISOR,558,4.69
3,DISTRITO,555,4.67
4,DIRECCION,99,0.83
5,ESTABLECIMIENTO,28,0.24
6,CODIGO,23,0.19
7,MUNICIPIO,23,0.19
8,DEPARTAMENTO,23,0.19
9,NIVEL,23,0.19


### 2.d Cantidad de valores únicos por variable

In [61]:
unicos = datos.drop(columns="__archivo_origen").nunique(dropna=True)
tabla_unicos = unicos.sort_values(ascending=False).reset_index()
tabla_unicos.columns = ["variable", "valores_unicos"]
display(tabla_unicos)

,variable,valores_unicos
0,CODIGO,11867
1,DIRECCION,7526
2,ESTABLECIMIENTO,6667
3,TELEFONO,6571
4,DIRECTOR,5561
5,DISTRITO,1681
6,SUPERVISOR,1285
7,MUNICIPIO,352
8,DEPARTAMENTAL,26
9,DEPARTAMENTO,23


### 2.e Registros duplicados exactos

In [62]:
cols_sin_origen = [c for c in datos.columns if c != "__archivo_origen"]
dup_exactos = datos.duplicated(subset=cols_sin_origen).sum()
dup_codigo = datos["CODIGO"].duplicated().sum()
tabla_duplicados = pd.DataFrame({
    "métrica": ["Registros duplicados exactos (ignorando __archivo_origen)", "Códigos (CODIGO) duplicados"],
    "valor": [dup_exactos, dup_codigo]
})
display(tabla_duplicados)

,métrica,valor
0,Registros duplicados exactos (ignorando __arch...,22
1,Códigos (CODIGO) duplicados,22
